In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip -q install idx2numpy tf2onnx

!wget -q https://biometrics.nist.gov/cs_links/EMNIST/gzip.zip
!unzip -q gzip.zip

!gunzip -f gzip/emnist-digits-train-images-idx3-ubyte.gz
!gunzip -f gzip/emnist-digits-train-labels-idx1-ubyte.gz
!gunzip -f gzip/emnist-digits-test-images-idx3-ubyte.gz
!gunzip -f gzip/emnist-digits-test-labels-idx1-ubyte.gz

In [ ]:
import idx2numpy
import numpy as np

X_train = idx2numpy.convert_from_file(
    "gzip/emnist-digits-train-images-idx3-ubyte"
)
y_train = idx2numpy.convert_from_file(
    "gzip/emnist-digits-train-labels-idx1-ubyte"
)

X_test = idx2numpy.convert_from_file(
    "gzip/emnist-digits-test-images-idx3-ubyte"
)
y_test = idx2numpy.convert_from_file(
    "gzip/emnist-digits-test-labels-idx1-ubyte"
)

print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)

In [ ]:
import numpy as np

X_train = np.rot90(X_train, k=-1, axes=(1,2))
X_train = np.flip(X_train, axis=2)

X_test = np.rot90(X_test, k=-1, axes=(1,2))
X_test = np.flip(X_test, axis=2)

X_train = X_train.astype("float32")/255.0
X_test = X_test.astype("float32")/255.0

X_train = X_train[...,None]
X_test = X_test[...,None]

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,4))

for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(X_train[i].squeeze(), cmap="gray")
    plt.title(int(y_train[i]))
    plt.axis("off")

plt.show()

In [ ]:
import tensorflow as tf

model = tf.keras.Sequential([

    tf.keras.layers.Input((28,28,1)),

    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomTranslation(0.08,0.08),
    tf.keras.layers.RandomZoom(0.10),

    tf.keras.layers.Conv2D(32,3,padding="same",activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(64,3,padding="same",activation="relu"),
    tf.keras.layers.MaxPooling2D(),

    tf.keras.layers.Conv2D(128,3,padding="same",activation="relu"),

    tf.keras.layers.Flatten(),

    tf.keras.layers.Dense(256,activation="relu"),
    tf.keras.layers.Dropout(0.4),

    tf.keras.layers.Dense(10,activation="softmax")

])

model.summary()

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
callbacks = [

    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=3,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2
    )

]

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=20,
    batch_size=128,
    callbacks=callbacks
)

In [ ]:
loss, acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", acc)

In [ ]:
model.save("/kaggle/working/digit_model.keras")

In [ ]:
!python -m tf2onnx.convert \
--keras /kaggle/working/digit_model.keras \
--output /kaggle/working/digit_model.onnx

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,4))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150)
plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns

preds = model.predict(X_test).argmax(axis=1)
cm = confusion_matrix(y_test, preds)

plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - EMNIST Digits')
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
import os
print(os.listdir("/kaggle/working"))